# Train a model

This notebook walks you through training your own model on the benchmark. To keep things quick, the demos here use just 50 species, so the results won't line up with the other benchmark baselines.

Here's the plan:

1. get to know the config system that controls training parameters
2. train a lightweight DeepSDM (MLP) on the demo data
3. train a Random Forest baseline on the same data
4. find out where the outputs land, and how to scale up to the full benchmark

> Keep in mind: the demo is illustrative. With 50 species, ~30 k cells, and range masking off, the metrics are weak by design.

In [2]:
import os
import rootutils
ROOT = rootutils.setup_root(os.path.abspath(""), indicator=".project-root", pythonpath=True, cwd=True)
os.environ["WANDB_MODE"] = "offline"          # no experiment-tracking account needed
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE" # play nice with networked filesystems
print("✅ ready — working from", ROOT)

✅ ready — working from /path/to/sdm_benchmark


## The demo data

`examples/demo_data/` is a real slice of SAGE. It uses the same HDF5 layout as the full dataset (targets, predictors, background, sPlotOpen), just cut down to 50 species:

In [3]:
import pandas as pd
species = pd.read_csv("examples/demo_data/targets/species_names.csv")
print(f"{len(species)} species, e.g.:", ", ".join(species.iloc[:5, 0].astype(str)))

total = 0
for root_, _, files in os.walk("examples/demo_data"):
    for f in sorted(files):
        total += os.path.getsize(os.path.join(root_, f))
print(f"demo_data total size: {total/1e6:.0f} MB")

50 species, e.g.: Acacia leiocalyx, Anemone trifolia, Anthosachne scabra, Anthyllis montana, Artemisia campestris
demo_data total size: 26 MB


## The config system

Instead of one large settings file, every run is assembled from small, reusable config layers. That way you can mix and match a dataset, a model, and a set of training options easily. A single experiment file names the pieces it wants and overrides only the few values it cares about.

The layers merge top to bottom, and each one overrides the layer above it:

| Layer | File(s) | What it contributes |
|---|---|---|
| Defaults | `local_default.yaml`, `default.yaml` | your machine's paths + framework-wide defaults |
| Data | `configs/data/<name>.yaml` | which dataset, predictor groups, batch size, range masking |
| Model | `configs/model/<name>.yaml` | the network, the loss, and the metrics |
| Experiment | `configs/experiments/<name>.yaml` | selects a data + model config and tweaks the run |
| CLI | `--key value` at the command line | last-word overrides, handy for sweeps |

You launch a run by naming its experiment, so `--exp demo` loads `configs/experiments/demo.yaml`. An experiment file is deliberately short. It points `data_config` and `model_config` at the layers to build on, then changes just what this run needs. The demo experiment says to use the demo data and the small-MLP model, repoints `data_dir` at the bundled subset, and sets a couple of training knobs (10 epochs, CPU). Everything else, like the predictor list, the loss weights, and the metrics, is inherited from the data and model layers it selected:

In [7]:
!cat configs/experiments/demo.yaml

/bin/bash: /opt/conda/envs/sdm/lib/libtinfo.so.6: no version information available (required by /bin/bash)
# ─────────────────────────────────────────────────────────────────────────────
# Demo experiment — trains a lightweight DeepSDM MLP on the bundled 50-species
# subset in examples/demo_data/, so the whole pipeline runs end-to-end on CPU
# without downloading the full SAGE data:
#
#     python src/main.py --exp demo fit
#
# Like every experiment, this file simply composes reusable config layers and
# overrides the few values this run actually needs.
# ─────────────────────────────────────────────────────────────────────────────

# Layers to build on: which data config and which model config to merge in.
data_config: demo          # configs/data/demo.yaml      — the demo dataset + loader
model_config: demo_mlp     # configs/model/demo_mlp.yaml — a small MLP + light loss

# Train on the bundled demo data instead of the full dataset from local_default.yaml.
# Path is relative to the r

## Train a DeepSDM (MLP)

The command below trains the model, then evaluates on the held-out sPlotOpen split (`test_after_fit`). On CPU the demo takes about 1–2 minutes:

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "src/main.py", "--exp", "demo", "fit",
     "--trainer.enable_progress_bar=false"],
    check=True,
)

Training wrote a checkpoint and a `test_predictions.h5` into a fresh run folder under `logs/`. That's the same prediction file the evaluation toolkit reads:

In [11]:
import glob
import h5py
import yaml
log_dir = yaml.safe_load(open("configs/local_default.yaml"))["log_dir"]
runs = sorted(glob.glob(os.path.join(log_dir, "**", "test_predictions.h5"), recursive=True), key=os.path.getmtime)
latest = runs[-1]
print("newest run:", os.path.dirname(latest))
with h5py.File(latest) as f:
    print("test_predictions.h5:", {k: f[k].shape for k in f.keys()})
ckpts = glob.glob(os.path.join(os.path.dirname(latest), "checkpoints", "*.ckpt"))
print("checkpoint:", os.path.basename(ckpts[0]) if ckpts else "(none)")

newest run: /path/to/logs/sage_benchmark/891tx40v
test_predictions.h5: {'locations': (24044, 2), 'probabilities': (24044, 50), 'targets': (24044, 50)}
checkpoint: epoch=09-val_auroc=76.1935.ckpt


## Train a Random Forest baseline

The traditional baselines train one model per species, so `sklearn_train.py` takes a species range `start end`. It's an inclusive slice by species index (`0 3` = species 0, 1, 2, 3), which makes it easy to split the 5,771 species across parallel jobs. Point it at the demo data with `--data-dir`. Here we fit species 0–3 (four species):

In [ ]:
import importlib.util
import subprocess, sys

# The single-species baselines live in the `baselines` extra (scikit-learn,
# imbalanced-learn, lightgbm, pygam) — not part of `.[tutorials]`.
if importlib.util.find_spec("imblearn") is None:
    print("Skipping: the Random Forest baseline needs the `baselines` extra.\n"
          'Install it with:  pip install -e ".[baselines]"')
else:
    subprocess.run(
        [sys.executable, "scripts/traditional_sdms/sklearn_train.py", "0", "3",
         "--config", "configs/model/randomforest.yaml",
         "--data-dir", "examples/demo_data/"],
        check=True,
    )

Re-running skips any species whose model file already exists (`skip_existing` in the config), so running the cell above again would train nothing. Add `--force` to retrain them anyway:

```bash
python scripts/traditional_sdms/sklearn_train.py 0 3 \
    --config configs/model/randomforest.yaml --data-dir examples/demo_data/ --force
```

Swap `randomforest.yaml` for `glm.yaml` / `gam.yaml` / `brt.yaml` to run the other baselines, and widen the range (e.g. `0 49`) for more species.

## Scaling up to the full benchmark

To train a comparable baseline, point at the full data and the full-size model:

- Data: download the processed data and set `data_dir` in `configs/local_default.yaml`.
- Predictors: choose which groups to feed via the `predictors_subset` in the data config (location / CHELSA / SoilGrids / topography / Human Footprint).
- Loss: `lambda_1` / `lambda_2` weight observations against background points. You can also enable per-species weighting and range masking (`po_range_mask` / `pa_range_mask`), both off in the demo and on for full runs.
- Checkpointing: monitor `val/auroc`.

## Where to next

- [03 · Evaluate Models](03_evaluate_models.ipynb): score `test_predictions.h5`, with metrics tables and the delta heatmap
- [04 · Generate Maps](04_generate_maps.ipynb): map a trained model